# PS-Q1: Biomedical Signal Classifier
## Part 3: Hardware Transpilation, Processor Benchmarking & Phase 2 Processor C Design

### Universal Challenge Framework - Variational Quantum Classifier (VQC) Track
**Objective:** Benchmark the trained biomedical VQC on NISQ hardware topologies (**Processors A & B**), evaluate architectural metrics (**CX count, Circuit Depth, SWAPs**), and design a **Phase 2 Processor C** with rigorous engineering justification and re-run comparison.

---
### Challenge Architecture Overview
In the Universal Challenge Framework:
- **Processor A:** 6-qubit Linear Chain ($0-1-2-3-4-5$). Limited 1D connectivity.
- **Processor B:** 6-qubit T-Junction / Heavy-Hex subsystem (Qubit 2 acts as central degree-3 branching hub).
- **Phase 2 Processor C (Custom Engineered):** Chordal Ring topology (Hexagon perimeter + diameter cross-connections).


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))

from src.vqc_classifier import BiomedicalVQC
from src.hardware_processors import (
    benchmark_circuit_on_processors,
    plot_processor_coupling_maps,
    PROCESSOR_REGISTRY
)
from src.evaluation import plot_hardware_benchmarks

print("Hardware modules loaded!")

### 1. Hardware Coupling Maps Visualization
Let's visualize the physical qubit connectivity graphs for Processors A, B, and C.


In [ ]:
plot_processor_coupling_maps()
plt.show()

### 2. Transpilation & Hardware Resource Analysis
We transpile the representative biomedical VQC circuit targeting each processor's coupling map using Qiskit's pass manager (optimization level 3, SabreSwap routing).


In [ ]:
# Create representative trained VQC circuit
vqc = BiomedicalVQC(num_qubits=6, num_layers=2)
dummy_x = np.array([np.pi/3, np.pi/4, np.pi/2, 2*np.pi/3, np.pi/6, 3*np.pi/4])
sample_qc = vqc.get_circuit_for_sample(dummy_x)

# Run hardware transpilation benchmark
hw_benchmarks = benchmark_circuit_on_processors(sample_qc, optimization_level=3, seed_transpiler=42)

# Display Circuit-Resource Table
resource_rows = []
for p_name, data in hw_benchmarks.items():
    resource_rows.append({
        "Processor": p_name,
        "Topology": data["topology"],
        "2-Qubit CX Count": data["cx_count"],
        "Circuit Depth": data["depth"],
        "SWAP Gates": data["swap_count"],
        "Total Gates": data["total_gates"]
    })

df_resources = pd.DataFrame(resource_rows)
df_resources

### 3. Hardware Tradeoff Comparison Bar Chart

In [ ]:
plot_hardware_benchmarks(hw_benchmarks)
plt.show()

### 4. Phase 2 Processor C Design & Engineering Justification

#### A. Architectural Bottlenecks on Processors A and B:
1. **Processor A (Linear Chain):**
   - The cyclic entangling ansatz requires closing the loop ($CX(5, 0)$).
   - In a 1D chain, interacting Qubit 5 with Qubit 0 requires traversing 4 intermediate hops ($5 \leftrightarrow 4 \leftrightarrow 3 \leftrightarrow 2 \leftrightarrow 1 \leftrightarrow 0$).
   - This necessitates multiple SWAP gates (each SWAP decomposes into 3 CX gates), inflating CX count and circuit depth by $>50\%$.
2. **Processor B (T-Topology):**
   - Qubit 2 acts as an information bottleneck. Interactions between peripheral nodes (e.g. Qubit 5 to 0 or 4) require multi-hop routing through Qubit 2.

#### B. Phase 2 Processor C Solution:
- **Design:** Engineered as a **Chordal Ring** with native ring perimeter couplings $(0,1), (1,2), (2,3), (3,4), (4,5), (5,0)$ and diameter shortcut bridges $(0,3), (1,4), (2,5)$.
- **Engineering Benefits:**
  - **Zero SWAP Gates:** The circular entangling topology of the VQC maps natively onto the perimeter ring with **0 routing SWAP overhead**.
  - **Substantially Lower Depth:** Circuit depth drops drastically, minimizing decoherence ($T_1, T_2$ relaxation).
  - **Error Suppression:** Eliminating extraneous CX gates directly reduces cumulative two-qubit gate errors ($E_{2Q} \approx 10^{-2}$ per CX), preserving quantum state fidelity for clinical screening.
